# Paper RAG 레이아웃·OCR 학습 (Colab)

이 노트북은 로컬 UI에서 내려받은 `paperrag-training-data.zip`을 사용합니다. **T4 GPU를 선택한 뒤 `런타임 → 모두 실행`을 한 번 누르세요.** 파일 선택 창에서 ZIP을 고르면 데이터 준비까지 자동 진행되고, 마지막 화면의 학습 버튼만 누르면 됩니다.

In [ ]:
#@title 1. GPU 확인 및 공식 학습 도구 설치
import os, subprocess, sys
assert subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL).returncode == 0, '런타임 유형을 T4 GPU로 변경하세요.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle-gpu', 'paddlex', 'paddleocr', 'pyyaml', 'ipywidgets'], check=True)
if not os.path.isdir('/content/PaddleOCR'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', '/content/PaddleOCR'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/PaddleOCR/requirements.txt'], check=True)
print('설치 완료')

In [ ]:
#@title 2. UI에서 받은 학습데이터 ZIP 선택
from google.colab import files
uploaded = files.upload()
archive_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive_name, 'paperrag-training-data.zip을 선택하세요.'
print('선택:', archive_name)

In [ ]:
#@title 3. 데이터 검사·학습 형식 변환
import hashlib, json, pathlib, shutil, zipfile
root=pathlib.Path('/content/paperrag_data')
if root.exists(): shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile('/content/'+archive_name) as z: z.extractall(root)
layout=[json.loads(x) for x in (root/'layout/annotations.jsonl').read_text().splitlines() if x.strip()]
ocr=[json.loads(x) for x in (root/'ocr/labels.jsonl').read_text().splitlines() if x.strip()]
classes=['title','author','abstract','section_header','text','table','table_caption','figure','figure_caption','formula','reference','header_footer']
cat={name:i+1 for i,name in enumerate(classes)}
def split(doc): return 'val' if int.from_bytes(hashlib.sha256(doc.encode()).digest()[:8],'big')/2**64 < .2 else 'train'
for part in ['train','val']:
 rows=[r for r in layout if split(r['document_id'])==part]; coco={'images':[],'annotations':[],'categories':[{'id':v,'name':k} for k,v in cat.items()]}; aid=1
 for iid,r in enumerate(rows,1):
  coco['images'].append({'id':iid,'file_name':r['image'].replace('layout/images/',''),'width':r['width'],'height':r['height']})
  for b in r['blocks']:
   x1,y1,x2,y2=b['bbox']; w=x2-x1; h=y2-y1
   if b['label'] in cat and w>0 and h>0: coco['annotations'].append({'id':aid,'image_id':iid,'category_id':cat[b['label']],'bbox':[x1,y1,w,h],'area':w*h,'iscrowd':0}); aid+=1
 (root/f'layout/{part}.json').write_text(json.dumps(coco,ensure_ascii=False))
 lines=[r['image'].replace('ocr/','')+'\t'+r['text'].replace('\n',' ').replace('\t',' ') for r in ocr if split(r['document_id'])==part]
 (root/f'ocr/{part}.txt').write_text('\n'.join(lines)+'\n')
print({'layout_pages':len(layout),'ocr_crops':len(ocr)})
assert layout or ocr, '승인 또는 교정된 영역이 없습니다.'

In [ ]:
#@title 4. 학습 버튼 (버튼을 한 번만 누르세요)
import glob, subprocess, sys, os, shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
out=widgets.Output(); model_root='/content/paperrag_models'; os.makedirs(model_root,exist_ok=True)
def config(patterns):
 for p in patterns:
  found=glob.glob('/content/PaddleOCR/'+p,recursive=True)
  if found: return sorted(found)[0]
 raise FileNotFoundError('공식 저장소에서 대응 config를 찾지 못했습니다. 노트북 버전과 PaddleOCR 버전을 확인하세요.')
def run_ocr():
 cfg=config(['configs/rec/**/*PP-OCRv5*mobile*rec*.yml','configs/rec/**/*PP-OCRv5*rec*.yml'])
 cmd=[sys.executable,'tools/train.py','-c',cfg,'-o',f'Global.save_model_dir={model_root}/ocr_train','Global.epoch_num=20',f'Train.dataset.data_dir={root}/ocr',f'Train.dataset.label_file_list=[{root}/ocr/train.txt]',f'Eval.dataset.data_dir={root}/ocr',f'Eval.dataset.label_file_list=[{root}/ocr/val.txt]']
 subprocess.run(cmd,cwd='/content/PaddleOCR',check=True)
 subprocess.run([sys.executable,'tools/export_model.py','-c',cfg,'-o',f'Global.pretrained_model={model_root}/ocr_train/best_accuracy',f'Global.save_inference_dir={model_root}/ocr'],cwd='/content/PaddleOCR',check=True)
def run_layout():
 try: cfg=config(['configs/**/*PP-DocLayout*.yml','configs/**/*layout*.yml'])
 except FileNotFoundError:
  import inspect, paddlex as pdx
  model=pdx.create_model('PP-DocLayout_plus-L'); candidates={'dataset_dir':str(root/'layout'),'output':f'{model_root}/layout_train','num_epochs':30,'batch_size':4,'device':'gpu'}
  params=inspect.signature(model.train).parameters; flexible=any(p.kind==inspect.Parameter.VAR_KEYWORD for p in params.values()); kwargs=candidates if flexible else {k:v for k,v in candidates.items() if k in params}
  model.train(**kwargs); return
 cmd=[sys.executable,'tools/train.py','-c',cfg,'-o',f'Global.save_model_dir={model_root}/layout_train','Global.epoch_num=30',f'Train.dataset.data_dir={root}/layout',f'Train.dataset.label_file_list=[{root}/layout/train.json]',f'Eval.dataset.data_dir={root}/layout',f'Eval.dataset.label_file_list=[{root}/layout/val.json]']
 subprocess.run(cmd,cwd='/content/PaddleOCR',check=True)
 subprocess.run([sys.executable,'tools/export_model.py','-c',cfg,'-o',f'Global.pretrained_model={model_root}/layout_train/best_accuracy',f'Global.save_inference_dir={model_root}/layout'],cwd='/content/PaddleOCR',check=True)
def execute(tasks):
 with out:
  clear_output(); print('학습 시작:',tasks)
  try:
   for task in tasks: run_layout() if task=='layout' else run_ocr()
   print('학습과 inference 모델 export 완료')
  except Exception as e: print('실패:',type(e).__name__,e); raise
b1=widgets.Button(description='레이아웃 학습',button_style='primary'); b2=widgets.Button(description='OCR 학습',button_style='primary'); b3=widgets.Button(description='둘 다 순서대로 학습',button_style='success')
b1.on_click(lambda _:execute(['layout'])); b2.on_click(lambda _:execute(['ocr'])); b3.on_click(lambda _:execute(['layout','ocr']))
display(widgets.HBox([b1,b2,b3]),out)

In [ ]:
#@title 5. 모델 ZIP 다운로드 버튼
from google.colab import files
import ipywidgets as widgets, shutil, os
def download(_):
 assert os.path.isdir('/content/paperrag_models'), '먼저 학습 버튼을 누르세요.'
 path=shutil.make_archive('/content/paperrag-trained-models','zip','/content/paperrag_models'); files.download(path)
button=widgets.Button(description='모델 ZIP 다운로드',button_style='success'); button.on_click(download); display(button)